# N072 · 树上路径与信息传递

**目标：** 区分向父亲延伸的路径和已完成的双侧路径。

**先修：** N071, N032。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 根到叶；任意路径；负贡献截断；前缀计数与回溯清理。

## 从问题到算法

树路径题首先区分根到叶、任意端点、以及只能向下三种路径。最大路径和向父节点只能返回一条分支，否则会分叉而不再是路径；但全局候选可同时接左右两条正贡献。向下路径计数用当前根路径的前缀频率，离开子树时必须撤销，避免把不同分支拼成假路径。

## 最小实现

**本章接口：** `has_path_sum(root, target)`、`max_path_sum(root)`、`path_sum_count(root, target)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root
def _postorder_nodes(root):
    stack=[(root,False)]
    while stack:
        node,done=stack.pop()
        if not node: continue
        if done: yield node
        else:
            stack.append((node,True)); stack.append((node.right,False)); stack.append((node.left,False))

from collections import Counter

def has_path_sum(root,target):
    stack=[(root,0)] if root else []
    while stack:
        node,prefix=stack.pop(); total=prefix+node.val
        if not node.left and not node.right and total==target: return True
        if node.right: stack.append((node.right,total))
        if node.left: stack.append((node.left,total))
    return False

def max_path_sum(root):
    if not root: raise ValueError('nonempty tree required')
    gain={None:0}; best=float('-inf')
    for node in _postorder_nodes(root):
        left=max(0,gain[node.left]); right=max(0,gain[node.right])
        best=max(best,node.val+left+right)
        gain[node]=node.val+max(left,right)
    return best

def path_sum_count(root,target):
    counts=Counter({0:1}); answer=0; stack=[(root,0,False)]
    while stack:
        node,prefix,leaving=stack.pop()
        if not node: continue
        if leaving:
            counts[prefix]-=1
            if counts[prefix]==0: del counts[prefix]
            continue
        total=prefix+node.val; answer+=counts[total-target]; counts[total]+=1
        stack.append((node,total,True)); stack.append((node.right,total,False)); stack.append((node.left,total,False))
    return answer

## 正确性、前提与复杂度

最大路径按唯一最高节点分类，向上返回值与全局候选承担不同语义；全负树仍至少取一个节点。前缀频率只含当前祖先路径，进入加一、退出减一维持范围。

**代价：** O(n)平均时间；最大和高度表O(n)空间；前缀计数工作空间O(h)，极端h=n。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([-10,9,20,None,None,15,7]); gain={None:0}; rows=[]
for n in _postorder_nodes(r):
    l=max(0,gain[n.left]); rr=max(0,gain[n.right]); gain[n]=n.val+max(l,rr)
    rows.append((n.val,gain[n],n.val+l+rr))
show_table(['节点','可向上传的单支贡献','在此结束的双支候选'],rows)

节点,可向上传的单支贡献,在此结束的双支候选
9,9,9
15,15,15
7,7,7
20,35,42
-10,25,34


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
r=tree_from_level([-10,9,20,None,None,15,7]); assert max_path_sum(r)==42
assert max_path_sum(tree_from_level([-3,-2,-5]))==-2
assert not has_path_sum(TreeNode(1,TreeNode(2)),1)
r=tree_from_level([10,5,-3,3,2,None,11,3,-2,None,1]); assert path_sum_count(r,8)==3
def from_start(node,target):
    if not node: return 0
    return int(node.val==target)+from_start(node.left,target-node.val)+from_start(node.right,target-node.val)
for target in range(-5,16):
    assert path_sum_count(r,target)==sum(from_start(n,target) for n in _postorder_nodes(r))
assert path_sum_count(None,0)==0
print('N072: 所有本章断言通过')

N072: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造不能把双臂路径继续向上传递的反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 清理前缀频次防止跨分支串线。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 112. Path Sum（canonical）
- 113. Path Sum II（canonical）
- 124. Binary Tree Maximum Path Sum（canonical）
- 437. Path Sum III（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。